#**Packages**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#**LU Factorization**

In [ ]:
def LU_factorization(A, method):

    if method == 1:
        """
        Computes the LU factorization (no pivoting) of a given dense nxn matrix A, stored as a 2D-array.
        Updates A so that the returned A contains L in the lower triangular part
        and U in the upper triangular part (including the diagonal).
        """

        n = len(A)      #Finds the dimension of A
        for i in range(n):
            alpha = A[i][i]     #Marks the diagonal element
            if alpha == 0:
                print("Error: This matrix can not be decomposed into L and U without pivoting due to a division by 0.")
                return None     #Exit the function completely if a 0 is encountered on the diagonal
            for k in range(i+1,n):
                A[k][i] = A[k][i]/alpha     #Updates each element in the column below the diagonal to its "lambda" value
                for j in range(i+1,n):
                    A[k][j] = A[k][j] - A[k][i] * A[i][j]       #Updates the elements in the submatrix to their new values based on the "lambda"

        return A  # Return the factorized matrix A



    elif method == 2:
        """
        Computes the LU factorization (partial pivoting) of a given dense nxn matrix A, stored as a 2D-array.
        Updates A so that the returned A contains L in the lower triangular part
        and U in the upper triangular part (including the diagonal).
        """

        n = len(A)  # Finds the dimension of A
        P = list(range(n))  # Initialize permutation vector to track row swaps

        for i in range(n):
            # Partial pivoting: Find the row with the largest pivot in column i
            max_row = i
            max_value = abs(A[i][i])

            for r in range(i+1, n):
                if abs(A[r][i]) > max_value:
                    max_value = abs(A[r][i])
                    max_row = r

            # Check if the matrix is singular (no non-zero pivot found)
            if max_value == 0:
                print(f"Error: Matrix is singular. No valid pivot in column {i}.")
                return None, P

            # Swap the current row with the max_row
            if max_row != i:
                A[i], A[max_row] = A[max_row], A[i]  # Swap rows in A
                P[i], P[max_row] = P[max_row], P[i]  # Swap rows in permutation vector (for consistency)

            # Pivot element
            alpha = A[i][i]
            if abs(alpha) < 1e-10:  # Handle small pivots (numerical stability)
                print(f"Error: Near-zero pivot encountered at row {i}.")
                return None, P

            # Perform the LU factorization updates
            for k in range(i+1, n):
                A[k][i] = A[k][i] / alpha  # Updates L's column
                for j in range(i+1, n):
                    A[k][j] -= A[k][i] * A[i][j]  # Updates the submatrix for U

        return A, P  # Return the LU factorized matrix A and the permutation vector P

#**Lower Triangular Solver (Forward Substitution)**

In [ ]:
def Lv_solver(LU, v):
    """
    Solves Lz = v for z, where L is the unit lower triangular part stored in LU.

    Parameters:
    LU (2D list): The matrix containing both the lower triangular matrix L
                  and the upper triangular matrix U.
    v (list): The vector v in the equation Lz = v.

    Returns:
    list: The solution vector z.
    """

    # Find the dimension of v
    n = len(v)

    # Initialize the solution vector z with zeros
    z = [0] * n

    # Solve the first element of z
    z[0] = v[0]

    # Loop over the elements of z from 2 to n (starting at index 1 in Python)
    for i in range(1, n):
        sum_prod = 0
        # Calculate the sum of products LU[i][j] * z[j] for j = 0 to i-1
        for j in range(i):
            sum_prod += LU[i][j] * z[j]

        # Solve for z[i]
        z[i] = v[i] - sum_prod

    return z

#**Upper Triangular Solver (Backward Substitution)**

In [ ]:
def Uv_solver(LU, v):
    """
    Solves Uz = v for z, where U is the upper triangular part stored in LU.

    Parameters:
    LU (2D list): The matrix containing both the lower triangular matrix L
                  and the upper triangular matrix U.
    v (list): The vector v in the equation Uz = v.

    Returns:
    list: The solution vector z.
    """

    # Find the dimension of v
    n = len(v)

    # Initialize the solution vector z with zeros
    z = [0] * n

    # Solve the last element of z
    z[n - 1] = v[n - 1] / LU[n - 1][n - 1]

    # Loop over the elements of z from n-1 down to 1
    for i in range(n - 2, -1, -1):
        sum_prod = 0
        # Calculate the sum of products LU[i][j] * z[j] for j = i+1 to n
        for j in range(i + 1, n):
            sum_prod += LU[i][j] * z[j]

        # Solve for z[i]
        z[i] = (v[i] - sum_prod) / LU[i][i]

    return z

#**Necessary Utility Functions**

In [ ]:
def compute_M_from_LU(LU):
    n = len(LU)
    M = [[0] * n for i in range(n)]

    # Perform matrix multiplication M = L * U
    for i in range(n):
        for j in range(n):
            if i == j:
                M[i][j] = LU[i][j]
                for k in range(i):
                    M[i][j] = M[i][j] + LU[i][k] * LU[k][i]
            elif i > j:
                for k in range(j+1):
                    M[i][j] = M[i][j] + LU[i][k] * LU[k][j]
            elif i < j:
                for k in range(i):
                    M[i][j] = M[i][j] + LU[i][k] * LU[k][j]
                M[i][j] = M[i][j] + LU [i][j]

    return M


def apply_permutation(A, P):
    # Initialize an empty list to store the permuted matrix
    PA = []

    # Loop through the permutation vector P and append the corresponding rows from A
    for i in P:
        PA.append(A[i])

    return PA


def generate_random_ULTM(n):
    # Create an n x n matrix filled with random integers in the specified range
    L_matrix = np.random.uniform(-0.99999, 0.99999, size=(n, n))
    # Set upper triangular part (excluding the diagonal) to zero
    for i in range(n):
        for j in range(i + 1, n):
            L_matrix[i][j] = 0
    # Set diagonal entries to 1
    for i in range(n):
        for j in range(n):
            if i==j:
                L_matrix[i][j] = 1
    return L_matrix


def generate_random_UTM(n, max_abs_value):
    # Create an n x n matrix with random elements between low and high
    U_matrix = np.random.uniform(-max_abs_value, max_abs_value + 1, size=(n, n))

    # Set lower triangular part (excluding the diagonal) to zero
    for i in range(n):
        for j in range(i):
            U_matrix[i][j] = 0

    # Adjust diagonal to have values between 1 and max_abs_value (excluding 0)
    for i in range(n):
        diag_val = 0
        while diag_val == 0 or (-1 < diag_val < 1):  # Exclude values between -1 and 1
            diag_val = np.random.uniform(-max_abs_value, max_abs_value)
        U_matrix[i][i] = diag_val

    U_matrix = U_matrix + (max_abs_value*3) * np.identity(n) #Add 3*max_abs_value times the identity to ensure diagonal dominance

    return U_matrix


def generate_random_x(n, max_abs_value):
    # Generate a random n x 1 vector with integer entries between low and high, stored as a 1D array
    vector = np.random.uniform(-max_abs_value, max_abs_value, size=(n, 1))
    return vector


def generate_random_P(n):
    #Generate a random permutation vector P
    P = np.random.permutation(n)
    P_list = P.tolist()
    return P_list


def matrix_copy(A):
    n = len(A)
    A_test_copy = [[0] * n for i in range(n)]
    for i in range(n):
        for j in range(n):
            A_test_copy[i][j] = A[i][j]
    return A_test_copy


def vector_difference(a, b):
    difference_vector =[]
    for i in range(len(a)):
        difference = a[i] - b[i]
        difference_vector.append(difference)
    return difference_vector


def euclidean_norm(vector):
    sum_of_squares = sum(x**2 for x in vector)
    return sum_of_squares ** 0.5


def matrix_difference(A, B):
    difference_matrix = []
    for i in range(len(A)):
        difference_row = []
        for j in range(len(A[i])):
            difference = float(A[i][j]) - float(B[i][j])
            difference_row.append(difference)
        difference_matrix.append(difference_row)
    return difference_matrix


def frobenius_norm(matrix):
    n = len(matrix)
    sum_of_squares = 0
    for i in range(n):
        for j in range(n):
            sum_of_squares = sum_of_squares + matrix[i][j] ** 2
    norm = sum_of_squares ** 0.5
    return norm


def calculate_mean(numbers):
    total_sum = sum(numbers)
    count = len(numbers)
    mean = total_sum / count
    return mean

#**General Tester**

In [ ]:
def main():
    method = int(input("Choose which method you'd like to use for the LU factorization. Choose 1 for no pivoting or choose 2 for partial pivoting: "))

    first_n = int(input("Enter the smallest size matrix to test: "))
    last_n = int(input("Enter the largest size matrix to test: "))
    trials = int(input("Enter the number of trials to run for each matrix size: "))
    max_abs_value = float(input("Enter an upper bound for the magnitude of the entries in the upper triangular matrix and the random x: "))


    factorization_error_mean_list = []
    solver_error_mean_list = []
    residual_error_mean_list = []
    growth_factor_mean_list = []
    for i in range(first_n, last_n+1):
        factorization_error_list = []
        solver_error_list = []
        residual_error_list = []
        growth_factor_list = []
        for j in range(trials):
            #Generate nonsingular L and U
            L_matrix = generate_random_ULTM(i)
            U_matrix = generate_random_UTM(i, max_abs_value)
            #print(L_matrix)
            #print(U_matrix)

            #Generate random x
            random_x = generate_random_x(i, max_abs_value)
            #print("Random x: " , random_x)


            if method == 1:

                #Compute A = L*U
                A_nptest = np.dot(L_matrix, U_matrix)
                #print(A_nptest)

                A_test = A_nptest.tolist()
                #print("\n Matrix A: " , A_test)
                print("\n Condition number: " , np.linalg.cond(A_test, p=None))

                #Copy A
                A_test_copy = matrix_copy(A_test)
                #print("\n A Copy: " , A_test_copy)


                #Factor A into L and U through routine
                B = LU_factorization(A_test, method)
                #print("\n Matrix B: " , B)

                #Compute C = L*U (from B)
                C = compute_M_from_LU(B)
                #print("\n Matrix C: " , C)

                #Find difference between A and C
                AC_difference = matrix_difference(A_test_copy,C)
                #print("\n Difference Matrix: " , AC_difference)

                #Find the factorization error using Frobenius norms
                error_top = frobenius_norm(AC_difference)
                #print(error_top)
                error_bottom = frobenius_norm(A_test)
                #print(error_bottom)

                factorization_error = error_top/error_bottom
                #print(factorization_error)

                factorization_error_list.append(factorization_error)



                #Compute Ax=b to find b
                A_np = np.array(A_test_copy)
                x_np = np.array(random_x)
                b = np.dot(A_np,x_np)
                b_list = []
                for k in range(len(b)):
                    b_list.append(b[k][0])
                #print("\n b: \n" , b_list)

                #Solve Ly=b for y
                y = Lv_solver(B,b_list)

                #Solve Ux=y for x
                x_comp = Uv_solver(B,y)
                #print("\n Computed x: \n" , x_comp)

                #Find difference between true x and computed x
                x_list=[]
                for k in range(len(random_x)):
                    x_list.append(random_x[k][0])
                x_x_comp_difference = vector_difference(x_list, x_comp)

                #Find the computed error
                error_top = euclidean_norm(x_x_comp_difference)
                #print(error_top)
                error_bottom = euclidean_norm(x_list)
                #print(error_bottom)

                solver_error = error_top/error_bottom
                #solver_error = np.log(error_top/error_bottom)
                #print(solver_error)

                solver_error_list.append(solver_error)

                #Find the residual
                x_comp_np = np.array(x_comp)
                A_x_comp = np.dot(A_np, x_comp_np)
                A_x_comp_list = []
                for k in range(len(A_x_comp)):
                    A_x_comp_list.append(A_x_comp[k])
                #print("\n A_x_comp: \n" , A_x_comp_list)

                residual_difference = vector_difference(b_list, A_x_comp_list)

                error_top = euclidean_norm(residual_difference)
                #print(error_top)
                error_bottom = euclidean_norm(b_list)
                #print(error_bottom)

                residual_error = error_top/error_bottom
                #print(residual_error)

                residual_error_list.append(residual_error)


                #Find the growth factor
                abs_B_np = np.abs(B)
                abs_B = abs_B_np.tolist()
                #print("\n abs_B: " , abs_B)
                abs_C = compute_M_from_LU(abs_B)
                #print("\n abs_C: " , abs_C)
                growth_top = frobenius_norm(abs_C)
                growth_bottom = frobenius_norm(A_test_copy)
                growth_factor = growth_top/growth_bottom
                growth_factor_list.append(growth_factor)



            elif method == 2:

                #Compute A = L*U
                A_nptest = np.dot(L_matrix, U_matrix)
                #print(A_nptest)

                A_test = A_nptest.tolist()
                #print("\n Matrix A: " , A_test)
                print("\n Condition number: " , np.linalg.cond(A_test, p=None))

                #Generate
                random_perm = generate_random_P(i)

                A_test_perm = apply_permutation(A_test, random_perm)

                #Copy A
                A_test_copy = matrix_copy(A_test_perm)
                #print("\n A Copy: " , A_test_copy)



                #Factor A into L and U through routine
                B, P = LU_factorization(A_test_perm, method)

                #Skips past cases where the threshold condition is violated
                if B is None:
                    print("One of the tests could not proceed.")
                    continue

                #print("\n Matrix B: " , B)
                #print("Permutation vector P is : " , P)

                #Compute C = L*U (from B)
                C = compute_M_from_LU(B)
                #print("\n Matrix C: " , C)

                #Permute A copy
                A_perm = apply_permutation(A_test_copy, P)

                #Find difference between the permuted A and C
                Aperm_C_difference = matrix_difference(A_perm,C)
                #print("\n Difference Matrix: " , Aperm_C_difference)

                #Find the  factorization error using Frobenius norms
                error_top = frobenius_norm(Aperm_C_difference)
                #print(error_top)
                error_bottom = frobenius_norm(A_test_copy)
                #print(error_bottom)

                factorization_error = error_top/error_bottom
                #print(factorization_error)

                factorization_error_list.append(factorization_error)


                #Compute Ax=b to find b
                A_np = np.array(A_test_copy)
                x_np = np.array(random_x)
                b = np.dot(A_np,x_np)
                b_list = []
                for k in range(len(b)):
                    b_list.append(b[k][0])
                #print("\n b: " , b_list)

                #correct = np.dot(np.linalg.inv(A_np) ,b)

                Pb = apply_permutation(b, P)
                #print("\n Pb: \n" , Pb)
                Pb_list = []
                # Iterate through each sublist (which is an array)
                for sublist in Pb:
                    # Iterate through each item in the sublist
                    for item in sublist:
                        # Append the item to the Pb_list after converting it to a float
                        Pb_list.append(float(item))
                #print("\n Pb_list: " , Pb_list)

                #Solve Ly=b for y
                y = Lv_solver(B,Pb_list)

                #Solve Ux=y for x
                x_comp = Uv_solver(B,y)
                #print("\n Computed x: \n" , x_comp)
                #print("CORRECT X : " , correct)

                #Find difference between true x and computed x
                x_list=[]
                for k in range(len(random_x)):
                    x_list.append(random_x[k][0])
                x_x_comp_difference = vector_difference(x_list, x_comp)
                #print("Difference vector: " , x_x_comp_difference)

                #Find the computed error
                error_top = euclidean_norm(x_x_comp_difference)
                #print(error_top)
                error_bottom = euclidean_norm(x_list)
                #print(error_bottom)

                solver_error = error_top/error_bottom
                #solver_error = np.log(error_top/error_bottom)

                #print(solver_error)

                solver_error_list.append(solver_error)


                #Find the residual
                x_comp_np = np.array(x_comp)
                PA_np = np.array(A_perm)
                PA_x_comp = np.dot(PA_np, x_comp_np)
                PA_x_comp_list = []
                for k in range(len(PA_x_comp)):
                    PA_x_comp_list.append(PA_x_comp[k])
                #print("\n PA_x_comp: \n" , PA_x_comp_list)

                residual_difference = vector_difference(Pb_list, PA_x_comp_list)

                error_top = euclidean_norm(residual_difference)
                #print(error_top)
                error_bottom = euclidean_norm(Pb_list)
                #print(error_bottom)

                residual_error = error_top/error_bottom
                #print(residual_error)

                residual_error_list.append(residual_error)


                #Find the growth factor
                abs_B_np = np.abs(B)
                abs_B = abs_B_np.tolist()
                #print("\n abs_B: " , abs_B)
                abs_C = compute_M_from_LU(abs_B)
                #print("\n abs_C: " , abs_C)
                growth_top = frobenius_norm(abs_C)
                growth_bottom = frobenius_norm(A_test_copy)
                growth_factor = growth_top/growth_bottom
                growth_factor_list.append(growth_factor)





        #print("FACTORIZATION: " , factorization_error_list)
        factorization_error_mean = calculate_mean(factorization_error_list)
        factorization_error_mean_list.append(factorization_error_mean)

        #print("SOLVER: " , solver_error_list)
        solver_error_mean = calculate_mean(solver_error_list)
        solver_error_mean_list.append(solver_error_mean)


        #print(residual_error_list)
        residual_error_mean = calculate_mean(residual_error_list)
        residual_error_mean_list.append(residual_error_mean)

        growth_factor_mean = calculate_mean(growth_factor_list)
        growth_factor_mean_list.append(growth_factor_mean)



    #print(factorization_error_mean_list)
    #print(solver_error_mean_list)
    #print(residual_error_mean_list)


    trial_sizes = list(range(first_n, last_n+1))

    plt.scatter(trial_sizes, factorization_error_mean_list)
    plt.title("Mean Factorization Error per Dimension")
    plt.xlabel("Dimension")
    plt.ylabel("Mean Factorization Error")
    plt.savefig("Factor_error_P170to200")
    plt.show()

    plt.scatter(trial_sizes, solver_error_mean_list)
    plt.title("Mean Solver Error per Dimension")
    plt.xlabel("Dimension")
    plt.ylabel("Mean Solver Error")
    plt.savefig("Solver_error_P170to200")
    plt.show()

    plt.scatter(trial_sizes, residual_error_mean_list)
    plt.title("Mean Residual per Dimension")
    plt.xlabel("Dimension")
    plt.ylabel("Mean Residual")
    plt.savefig("Residual_P170to200")
    plt.show()


    plt.scatter(trial_sizes, growth_factor_mean_list)
    plt.title("Mean Growth Factor per Dimension")
    plt.xlabel("Dimension")
    plt.ylabel("Mean Growth Factor")
    plt.savefig("Growth_factor_P170to200")
    plt.show()



main()

#**Empirical Tests**

In [ ]:
#EMPIRICAL TEST 1

Test1_A1_a = [[1,0,0,0,0],
              [0,2,0,0,0],
              [0,0,3,0,0],
              [0,0,0,4,0],
              [0,0,0,0,5]]

Test1_A1_b = [[1,0,0,0,0],
              [0,2,0,0,0],
              [0,0,3,0,0],
              [0,0,0,4,0],
              [0,0,0,0,5]]

Test1_A2_a = [[5,0,0,0,0],
              [0,4,0,0,0],
              [0,0,3,0,0],
              [0,0,0,2,0],
              [0,0,0,0,1]]

Test1_A2_b = [[5,0,0,0,0],
              [0,4,0,0,0],
              [0,0,3,0,0],
              [0,0,0,2,0],
              [0,0,0,0,1]]

print("\n Test 1 (A1): ")
#No pivoting
print("\n With no pivoting: ")
growth_bottom1 = frobenius_norm(Test1_A1_a)
Q1 = LU_factorization(Test1_A1_a,1)
if Q1 is not None:
    for row in Q1:
        print(row)
    abs_Q1_np = np.abs(Q1)
    abs_Q1 = abs_Q1_np.tolist()
    #print("\n abs_Q1: " , abs_Q1)
    abs_R1 = compute_M_from_LU(abs_Q1)
    #print("\n abs_R1: " , abs_R1)
    growth_top1 = frobenius_norm(abs_R1)
    growth_factor1 = growth_top1/growth_bottom1
    print("Growth Factor: " , growth_top1 , "/" , growth_bottom1 , "=" , growth_factor1)
else:
    print("Not possible")
#Partial pivoting
growth_bottom2 = frobenius_norm(Test1_A1_b)
Q2, P_1 = LU_factorization(Test1_A1_b,2)
print("\n With partial pivoting: ")
if Q2 is not None:
    for row in Q2:
        print(row)
    print("Permutation Vector: " , P_1)
    abs_Q2_np = np.abs(Q2)
    abs_Q2 = abs_Q2_np.tolist()
    #print("\n abs_Q2: " , abs_Q2)
    abs_R2 = compute_M_from_LU(abs_Q2)
    #print("\n abs_R2: " , abs_R2)
    growth_top2 = frobenius_norm(abs_R2)
    growth_factor2 = growth_top2/growth_bottom2
    print("Growth Factor: " , growth_top2 , "/" , growth_bottom2 , "=" , growth_factor2)
else:
    print("Not possible")

print("\n Test 1 (A2): ")
#No pivoting
print("\n With no pivoting: ")
growth_bottom3 = frobenius_norm(Test1_A2_a)
Q3 = LU_factorization(Test1_A2_a,1)
if Q3 is not None:
    for row in Q3:
        print(row)
    abs_Q3_np = np.abs(Q3)
    abs_Q3 = abs_Q3_np.tolist()
    #print("\n abs_Q3: " , abs_Q3)
    abs_R3 = compute_M_from_LU(abs_Q3)
    #print("\n abs_R3: " , abs_R3)
    growth_top3 = frobenius_norm(abs_R3)
    growth_factor3 = growth_top3/growth_bottom3
    print("Growth Factor: " , growth_top3 , "/" , growth_bottom3 , "=" , growth_factor3)
else:
    print("Not possible")
#Partial pivoting
growth_bottom4 = frobenius_norm(Test1_A2_b)
Q4, P_2 = LU_factorization(Test1_A2_b,2)
print("\n With partial pivoting: ")
if Q4 is not None:
    for row in Q4:
        print(row)
    print("Permutation Vector: " , P_2)
    abs_Q4_np = np.abs(Q4)
    abs_Q4 = abs_Q4_np.tolist()
    #print("\n abs_Q4: " , abs_Q4)
    abs_R4 = compute_M_from_LU(abs_Q4)
    #print("\n abs_R4: " , abs_R4)
    growth_top4 = frobenius_norm(abs_R4)
    growth_factor4 = growth_top4/growth_bottom4
    print("Growth Factor: " , growth_top4 , "/" , growth_bottom4 , "=" , growth_factor4)
else:
    print("Not possible")



#EMPIRICAL TEST 2

Test2_A1_a = [[0,0,0,0,1],
              [0,0,0,2,0],
              [0,0,3,0,0],
              [0,4,0,0,0],
              [5,0,0,0,0]]

Test2_A1_b = [[0,0,0,0,1],
              [0,0,0,2,0],
              [0,0,3,0,0],
              [0,4,0,0,0],
              [5,0,0,0,0]]

Test2_A2_a = [[0,0,0,0,5],
              [0,0,0,4,0],
              [0,0,3,0,0],
              [0,2,0,0,0],
              [1,0,0,0,0]]

Test2_A2_b = [[0,0,0,0,5],
              [0,0,0,4,0],
              [0,0,3,0,0],
              [0,2,0,0,0],
              [1,0,0,0,0]]

print("\n Test 2 (A1): ")
#No pivoting
print("\n With no pivoting: ")
growth_bottom5 = frobenius_norm(Test2_A1_a)
Q5 = LU_factorization(Test2_A1_a,1)
if Q5 is not None:
    for row in Q5:
        print(row)
    abs_Q5_np = np.abs(Q5)
    abs_Q5 = abs_Q5_np.tolist()
    #print("\n abs_Q5: " , abs_Q5)
    abs_R5 = compute_M_from_LU(abs_Q5)
    #print("\n abs_R5: " , abs_R5)
    growth_top5 = frobenius_norm(abs_R5)
    growth_factor5 = growth_top5/growth_bottom5
    print("Growth Factor: " , growth_top5 , "/" , growth_bottom5 , "=" , growth_factor5)
else:
    print("Not possible")
#Partial pivoting
growth_bottom6 = frobenius_norm(Test2_A1_b)
Q6, P_3 = LU_factorization(Test2_A1_b,2)
print("\n With partial pivoting: ")
if Q6 is not None:
    for row in Q6:
        print(row)
    print("Permutation Vector: " , P_3)
    abs_Q6_np = np.abs(Q6)
    abs_Q6 = abs_Q6_np.tolist()
    #print("\n abs_Q6: " , abs_Q6)
    abs_R6 = compute_M_from_LU(abs_Q6)
    #print("\n abs_R6: " , abs_R6)
    growth_top6 = frobenius_norm(abs_R6)
    growth_factor6 = growth_top6/growth_bottom6
    print("Growth Factor: " , growth_top6 , "/" , growth_bottom6 , "=" , growth_factor6)
else:
    print("Not possible")

print("\n Test 2 (A2): ")
#No pivoting
print("\n With no pivoting: ")
growth_bottom7 = frobenius_norm(Test2_A2_a)
Q7 = LU_factorization(Test2_A2_a,1)
if Q7 is not None:
    for row in Q7:
        print(row)
    abs_Q7_np = np.abs(Q7)
    abs_Q7 = abs_Q7_np.tolist()
    #print("\n abs_Q7: " , abs_Q7)
    abs_R7 = compute_M_from_LU(abs_Q7)
    #print("\n abs_R7: " , abs_R7)
    growth_top7 = frobenius_norm(abs_R7)
    growth_factor7 = growth_top7/growth_bottom7
    print("Growth Factor: " , growth_top7 , "/" , growth_bottom7 , "=" , growth_factor7)
else:
    print("Not possible")
#Partial pivoting
growth_bottom8 = frobenius_norm(Test2_A2_b)
Q8, P_4 = LU_factorization(Test2_A2_b,2)
print("\n With partial pivoting: ")
if Q8 is not None:
    for row in Q8:
        print(row)
    print("Permutation Vector: " , P_4)
    abs_Q8_np = np.abs(Q8)
    abs_Q8 = abs_Q8_np.tolist()
    #print("\n abs_Q8: " , abs_Q8)
    abs_R8 = compute_M_from_LU(abs_Q8)
    #print("\n abs_R8: " , abs_R8)
    growth_top8 = frobenius_norm(abs_R8)
    growth_factor8 = growth_top8/growth_bottom8
    print("Growth Factor: " , growth_top8 , "/" , growth_bottom8 , "=" , growth_factor8)
else:
    print("Not possible")



#EMPIRICAL TEST 3

Test3_A_a = [[1,0,0,0,6],
             [0,2,0,7,0],
             [0,0,3,0,0],
             [0,8,0,4,0],
             [9,0,0,0,5]]

Test3_A_b = [[1,0,0,0,6],
             [0,2,0,7,0],
             [0,0,3,0,0],
             [0,8,0,4,0],
             [9,0,0,0,5]]

print("\n Test 3: ")
#No pivoting
print("\n With no pivoting: ")
Q9 = LU_factorization(Test3_A_a,1)
if Q9 is not None:
    for row in Q9:
        print(row)
else:
    print("Not possible")
#Partial pivoting
Q10, P_5 = LU_factorization(Test3_A_b,2)
print("\n With partial pivoting: ")
if Q10 is not None:
    for row in Q10:
        print(row)
    print("Permutation Vector: " , P_5)
else:
    print("Not possible")



#EMPIRICAL TEST 4

Test4_A_a = [[1,0,0,0,0],
             [0.1,1,0,0,0],
             [-0.2,0.5,1,0,0],
             [0.7,-0.6,0.4,1,0],
             [0.8,-0.9,0.3,-0.2,1]]

Test4_A_b = [[1,0,0,0,0],
             [0.1,1,0,0,0],
             [-0.2,0.5,1,0,0],
             [0.7,-0.6,0.4,1,0],
             [0.8,-0.9,0.3,-0.2,1]]

print("\n Test 4: ")
#No pivoting
print("\n With no pivoting: ")
Q11 = LU_factorization(Test4_A_a,1)
if Q11 is not None:
    for row in Q11:
        print(row)
else:
    print("Not possible")
#Partial pivoting
Q12, P_6 = LU_factorization(Test4_A_b,2)
print("\n With partial pivoting: ")
if Q12 is not None:
    for row in Q12:
        print(row)
    print("Permutation Vector: " , P_6)
else:
    print("Not possible")



#EMPIRICAL TEST 5

Test5_A_a = [[2,0,0,0,0],
             [3,2,0,0,0],
             [4,3,2,0,0],
             [5,4,3,2,0],
             [6,5,4,3,2]]

Test5_A_b = [[2,0,0,0,0],
             [3,2,0,0,0],
             [4,3,2,0,0],
             [5,4,3,2,0],
             [6,5,4,3,2]]

print("\n Test 5: ")
#No pivoting
print("\n With no pivoting: ")
Q13 = LU_factorization(Test5_A_a,1)
if Q13 is not None:
    for row in Q13:
        print(row)
else:
    print("Not possible")
#Partial pivoting
Q14, P_7 = LU_factorization(Test5_A_b,2)
print("\n With partial pivoting: ")
if Q14 is not None:
    for row in Q14:
        print(row)
    print("Permutation Vector: " , P_7)
else:
    print("Not possible")



#EMPIRICAL TEST 6

Test6_A_a = [[14,4,0,0,0],
             [3,17,7,0,0],
             [0,6,12,1,0],
             [0,0,4,18,5],
             [0,0,0,8,13]]

Test6_A_b = [[14,4,0,0,0],
             [3,17,7,0,0],
             [0,6,12,1,0],
             [0,0,4,18,5],
             [0,0,0,8,13]]

print("\n Test 6: ")
#No pivoting
print("\n With no pivoting: ")
Q15 = LU_factorization(Test6_A_a,1)
if Q15 is not None:
    for row in Q15:
        print(row)
else:
    print("Not possible")
#Partial pivoting
Q16, P_8 = LU_factorization(Test6_A_b,2)
print("\n With partial pivoting: ")
if Q16 is not None:
    for row in Q16:
        print(row)
    print("Permutation Vector: " , P_8)
else:
    print("Not possible")



#EMPIRICAL TEST 7

Test7_A_a = [[1,0,0,0,1],
             [-1,1,0,0,1],
             [-1,-1,1,0,1],
             [-1,-1,-1,1,1],
             [-1,-1,-1,-1,1]]

Test7_A_b = [[1,0,0,0,1],
             [-1,1,0,0,1],
             [-1,-1,1,0,1],
             [-1,-1,-1,1,1],
             [-1,-1,-1,-1,1]]

print("\n Test 7: ")
#No pivoting
print("\n With no pivoting: ")
growth_bottom17 = frobenius_norm(Test7_A_a)
Q17 = LU_factorization(Test7_A_a,1)
if Q17 is not None:
    for row in Q17:
        print(row)
    abs_Q17_np = np.abs(Q17)
    abs_Q17 = abs_Q17_np.tolist()
    #print("\n abs_Q17: " , abs_Q17)
    abs_R17 = compute_M_from_LU(abs_Q17)
    #print("\n abs_R17: " , abs_R17)
    growth_top17 = frobenius_norm(abs_R17)
    growth_factor17 = growth_top17/growth_bottom17
    print("Growth Factor: " , growth_top17 , "/" , growth_bottom17 , "=" , growth_factor17)
else:
    print("Not possible")
#Partial pivoting
growth_bottom18 = frobenius_norm(Test7_A_b)
Q18, P_9 = LU_factorization(Test7_A_b,2)
print("\n With partial pivoting: ")
if Q18 is not None:
    for row in Q18:
        print(row)
    print("Permutation Vector: " , P_9)
    abs_Q18_np = np.abs(Q18)
    abs_Q18 = abs_Q18_np.tolist()
    #print("\n abs_Q18: " , abs_Q18)
    abs_R18 = compute_M_from_LU(abs_Q18)
    #print("\n abs_R18: " , abs_R18)
    growth_top18 = frobenius_norm(abs_R18)
    growth_factor18 = growth_top18/growth_bottom18
    print("Growth Factor: " , growth_top18 , "/" , growth_bottom18 , "=" , growth_factor18)
else:
    print("Not possible")



#EMPIRICAL TEST 8

Test8_A_a = [[9,15,21,12,3],
             [15,89,59,92,37],
             [21,59,94,61,61],
             [12,92,61,102,59],
             [3,37,61,59,183]]

print("\n Test 8: ")
#No pivoting
print("\n With no pivoting: ")
Q19 = LU_factorization(Test8_A_a,1)
if Q19 is not None:
    for row in Q19:
        print(row)
else:
    print("Not possible")
